## Exercise:

Load the cancer dataset and choose the best classification algorithm with the best hyperparameters.

- Define X and y

- To simplify, remove missing values

- Split data to train and test

- Use 5 fold cross validation and grid search on train data

- Choose appropriate validation metric

- Set grid parameters for each classification algorithm

- Build the best models for each classification algorithm according to the best estimator (best hyperparameters) given by the grid search

- Compare the performance of the algorithms with the best hyperparametrs on the test data according to confusion matrix, recall, precision, F1, and auc metrics

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# Load the dataset
cancer_df = pd.read_csv('../data/breast_cancer_wisconsin.csv')
print('Dataset shape:', cancer_df.shape)
print('Columns:', cancer_df.columns.tolist())
print('\nMissing values by column:')
print(cancer_df.isnull().sum())

cancer_df.head()

Dataset shape: (699, 11)
Columns: ['Id', 'Cl.thickness', 'Cell.size', 'Cell.shape', 'Marg.adhesion', 'Epith.c.size', 'Bare.nuclei', 'Bl.cromatin', 'Normal.nucleoli', 'Mitoses', 'Class']

Missing values by column:
Id                  0
Cl.thickness        0
Cell.size           0
Cell.shape          0
Marg.adhesion       0
Epith.c.size        0
Bare.nuclei        16
Bl.cromatin         0
Normal.nucleoli     0
Mitoses             0
Class               0
dtype: int64


,Id,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0


In [2]:
# Define X and y, remove missing values, and drop non-feature columns
cancer_df = cancer_df.dropna().copy()

if 'Class' in cancer_df.columns:
    target_column = 'Class'
elif 'diagnosis' in cancer_df.columns:
    target_column = 'diagnosis'
else:
    target_column = cancer_df.columns[-1]

X = cancer_df.drop(columns=['id', 'Id', 'ID', target_column], errors='ignore')
y = cancer_df[target_column].copy()

# Map target labels to 0/1 if needed (common in this breast cancer dataset)
if set(y.unique()) == {2, 4}:
    y = y.map({2: 0, 4: 1})

print('Target column:', target_column)
print('Feature matrix shape:', X.shape)
print('Target distribution:')
print(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print('\nTrain shape:', X_train.shape, y_train.shape)
print('Test shape:', X_test.shape, y_test.shape)

Target column: Class
Feature matrix shape: (683, 9)
Target distribution:
Class
0    444
1    239
Name: count, dtype: int64

Train shape: (546, 9) (546,)
Test shape: (137, 9) (137,)


In [3]:
# Configure pipelines and hyperparameter grids
pipelines = {
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier())
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(random_state=42))
    ]),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, solver='liblinear', random_state=42))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(probability=True, random_state=42))
    ])
}

param_grids = {
    'KNN': {
        'clf__n_neighbors': [3, 5, 7, 9],
        'clf__weights': ['uniform', 'distance'],
        'clf__p': [1, 2]
    },
    'Decision Tree': {
        'clf__max_depth': [None, 3, 5, 7, 9],
        'clf__min_samples_split': [2, 5, 10],
        'clf__criterion': ['gini', 'entropy']
    },
    'Logistic Regression': {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__penalty': ['l2']
    },
    'SVM': {
        'clf__C': [0.1, 1, 10],
        'clf__kernel': ['linear', 'rbf'],
        'clf__gamma': ['scale', 'auto']
    }
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metric = 'roc_auc'

best_models = {}
summary = []

for name, pipeline in pipelines.items():
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        cv=cv,
        scoring=scoring_metric,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_
    summary.append({
        'Model': name,
        'Best Score': grid.best_score_,
        'Best Params': grid.best_params_
    })
    print(f"{name} best {scoring_metric}: {grid.best_score_:.4f}")
    print(f"{name} best params: {grid.best_params_}\n")

pd.DataFrame(summary)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
KNN best roc_auc: 0.9942
KNN best params: {'clf__n_neighbors': 9, 'clf__p': 2, 'clf__weights': 'distance'}

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Decision Tree best roc_auc: 0.9752
Decision Tree best params: {'clf__criterion': 'entropy', 'clf__max_depth': 3, 'clf__min_samples_split': 2}

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Logistic Regression best roc_auc: 0.9972
Logistic Regression best params: {'clf__C': 0.1, 'clf__penalty': 'l2'}

Fitting 5 folds for each of 12 candidates, totalling 60 fits


c:\Users\TM39417\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


SVM best roc_auc: 0.9974
SVM best params: {'clf__C': 1, 'clf__gamma': 'scale', 'clf__kernel': 'linear'}



c:\Users\TM39417\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Best Score,Best Params
0,KNN,0.994226,"{'clf__n_neighbors': 9, 'clf__p': 2, 'clf__wei..."
1,Decision Tree,0.975153,"{'clf__criterion': 'entropy', 'clf__max_depth'..."
2,Logistic Regression,0.997231,"{'clf__C': 0.1, 'clf__penalty': 'l2'}"
3,SVM,0.997364,"{'clf__C': 1, 'clf__gamma': 'scale', 'clf__ker..."


In [5]:
# Compare best models on the test set using confusion matrix, precision, recall, f1, and AUC
results = []

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    results.append({
        'Model': name,
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC AUC': roc_auc_score(y_test, y_score),
        'Confusion Matrix': confusion_matrix(y_test, y_pred)
    })

results_df = pd.DataFrame([{k: v for k, v in row.items() if k != 'Confusion Matrix'} for row in results])
print(results_df)



                 Model  Precision    Recall        F1   ROC AUC
0                  KNN   0.920000  0.958333  0.938776  0.982912
1        Decision Tree   0.914894  0.895833  0.905263  0.978699
2  Logistic Regression   0.921569  0.979167  0.949495  0.992743
3                  SVM   0.920000  0.958333  0.938776  0.991807


In [6]:
for row in results:
    print(f"\n{name if False else row['Model']}")
    print('Confusion matrix:')
    print(row['Confusion Matrix'])
    print('Classification report:')
    print(classification_report(y_test, best_models[row['Model']].predict(X_test), digits=4))


KNN
Confusion matrix:
[[85  4]
 [ 2 46]]
Classification report:
              precision    recall  f1-score   support

           0     0.9770    0.9551    0.9659        89
           1     0.9200    0.9583    0.9388        48

    accuracy                         0.9562       137
   macro avg     0.9485    0.9567    0.9523       137
weighted avg     0.9570    0.9562    0.9564       137


Decision Tree
Confusion matrix:
[[85  4]
 [ 5 43]]
Classification report:
              precision    recall  f1-score   support

           0     0.9444    0.9551    0.9497        89
           1     0.9149    0.8958    0.9053        48

    accuracy                         0.9343       137
   macro avg     0.9297    0.9254    0.9275       137
weighted avg     0.9341    0.9343    0.9341       137


Logistic Regression
Confusion matrix:
[[85  4]
 [ 1 47]]
Classification report:
              precision    recall  f1-score   support

           0     0.9884    0.9551    0.9714        89
           1    